In [1]:
# Cargar datos

import sys
sys.path.append("..")     # para poder importar desde src/

from src.data import cargar_enemdu, indicadores_laborales
import pandas as pd

df = cargar_enemdu()
print(df.shape)
df.info(memory_usage="deep")
df.head()

(82894, 22)
<class 'pandas.DataFrame'>
RangeIndex: 82894 entries, 0 to 82893
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area          82894 non-null  float64
 1   ciudad        82894 non-null  float64
 2   conglomerado  82894 non-null  str    
 3   p02           82894 non-null  float64
 4   p03           82894 non-null  float64
 5   p10a          78757 non-null  float64
 6   p10b          76439 non-null  float64
 7   p15           78757 non-null  float64
 8   p24           40298 non-null  float64
 9   p51a          40298 non-null  float64
 10  p51b          2413 non-null   float64
 11  p51c          2412 non-null   float64
 12  estrato       82894 non-null  str    
 13  fexp          82894 non-null  float64
 14  dominio       82894 non-null  float64
 15  nnivins       78757 non-null  float64
 16  ingrl         35870 non-null  float64
 17  condact       82894 non-null  float64
 18  secemp        39920 non-n

,area,ciudad,conglomerado,p02,p03,p10a,p10b,p15,p24,p51a,...,estrato,fexp,dominio,nnivins,ingrl,condact,secemp,grupo1,rama1,upm
0,1.0,10150.0,000801,2.0,69.0,9.0,2.0,6.0,NaN,NaN,...,2712,46.2024,3.0,5.0,NaN,9.0,NaN,NaN,NaN,010150000801
1,1.0,10150.0,000801,1.0,28.0,9.0,3.0,6.0,40.0,40.0,...,2712,46.2024,3.0,5.0,526.0,1.0,1.0,5.0,18.0,010150000801
2,1.0,10150.0,000801,2.0,19.0,9.0,1.0,6.0,NaN,NaN,...,2712,46.2024,3.0,5.0,NaN,9.0,NaN,NaN,NaN,010150000801
3,1.0,10150.0,000801,2.0,87.0,4.0,6.0,6.0,NaN,NaN,...,2712,46.2024,3.0,3.0,NaN,9.0,NaN,NaN,NaN,010150000801
4,1.0,10150.0,000801,2.0,65.0,9.0,5.0,6.0,40.0,40.0,...,2712,46.2024,3.0,5.0,800.0,1.0,1.0,7.0,6.0,010150000801


In [2]:
# Sanity check

print("Nulos por columna:")
print(df.isna().sum())

print("\nSuma de fexp (poblacion represent.):", f"{df['fexp'].sum():,.0f}")
print("\nDistribucion de condact (ponderada, en %):")
print((df.groupby("condact")["fexp"].sum() / df["fexp"].sum() * 100).round(2))

Nulos por columna:
area                0
ciudad              0
conglomerado        0
p02                 0
p03                 0
p10a             4137
p10b             6455
p15              4137
p24             42596
p51a            42596
p51b            80481
p51c            80482
estrato             0
fexp                0
dominio             0
nnivins          4137
ingrl           47024
condact             0
secemp          42974
grupo1          42974
rama1           42974
upm                 0
dtype: int64

Suma de fexp (poblacion represent.): 19,026,779

Distribucion de condact (ponderada, en %):
condact
0.0    28.93
1.0    16.16
2.0     7.98
3.0     0.36
4.0    14.79
5.0     4.38
6.0     0.10
7.0     1.36
8.0     0.17
9.0    25.77
Name: fexp, dtype: float64


In [ ]:
# Primera tabla de validacion

oficial = {"Participacion global": 63.8, "Empleo adecuado": 35.7,
            "Subempleo": 18.4, "Desempleo": 3.4}

calc = indicadores_laborales(df)
comparacion = pd.DataFrame({"Calculado": calc, "Oficial INEC": oficial}).round(1)
comparacion["Diferencia"] = (comparacion["Calculado"] - comparacion["Oficial INEC"]).round(1)
comparacion

,Calculado,Oficial INEC,Diferencia
Participacion global,63.7,63.8,-0.1
Empleo adecuado,35.7,35.7,0.0
Subempleo,18.4,18.4,0.0
Desempleo,3.4,3.4,0.0


In [4]:
# Variables que faltan

import pyreadstat
_, meta = pyreadstat.read_sav("../data/raw/spss/enemdu_persona_2026_l_trimestre.sav", metadataonly=True) 

for clave in ["prov", "esc", "anios", "horas", "p24", "p51", "inf", "etnia", "p15"]:
    hallazgos = [(c, meta.column_names_to_labels.get(c)) for c in meta.column_names if clave in c.lower()]
    print(clave, "->", hallazgos[:6])

prov -> []
esc -> []
anios -> []
horas -> []
p24 -> [('p24', 'Horas de trabajo en la semana anterior')]
p51 -> [('p51a', 'Horas de trabajo principal'), ('p51b', 'Horas de trabajo secundario'), ('p51c', 'Horas de otros trabajos')]
inf -> [('cod_inf', 'Código del Informante')]
etnia -> []
p15 -> [('p15', 'Como se considera'), ('p15aa', 'Dónde nació'), ('p15ab', 'Lugar de nacimiento')]


In [ ]:
# Comprobacion diferencia 0.1 en participacion (comprobacion de PET)

c = df["condact"]
pea = c.between(1, 8)

pet_condact = c.between(1, 9)      # lo que usamos
pet_edad = df["p03"] >= 15         # alternativa por edad

for nombre, pet in [("PET por condact", pet_condact), ("PET por edad >= 15", pet_edad)]:
    tasa = 100 * df.loc[pea, "fexp"].sum() / df.loc[pet, "fexp"].sum()
    print(f"{nombre}: {tasa:.3f}")

PET por condact: 63.745
PET por edad >= 15: 63.745


In [6]:
# Completar variables del diccionario

import pyreadstat
ruta = "../data/raw/spss/enemdu_persona_2026_l_trimestre.sav"
_, meta = pyreadstat.read_sav(ruta, metadataonly=True)

print("Códigos de p15:", meta.variable_value_labels.get("p15"), "\n")

for clave in ["p10", "p07", "p41", "reg", "dominio", "estrato", "upm", "instru"]:
    hallazgos = [(c, meta.column_names_to_labels.get(c)) for c in meta.column_names if clave in c.lower()]
    print(clave, "->", hallazgos[:6])

d, _ = pyreadstat.read_sav(ruta, usecols=["ciudad", "conglomerado"])
a_texto = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
for col in ["ciudad", "conglomerado"]:
    print(f"\n{col}: prefijos de 2 dígitos ->", sorted(a_texto(d[col]).str[:2].unique()))

Códigos de p15: {1.0: ' Indígena', 2.0: ' Afroecuatoriano', 3.0: ' Negro', 4.0: ' Mulato', 5.0: ' Montuvio', 6.0: ' Mestizo', 7.0: ' Blanco', 8.0: ' Otra'} 

p10 -> [('p10a', 'Nivel de instrucción'), ('p10b', 'Año aprobado')]
p07 -> [('p07', 'Asiste a clases')]
p41 -> [('p41', 'Grupo de ocupación')]
reg -> []
dominio -> [('dominio', 'Dominios')]
estrato -> [('estrato', 'Estratos')]
upm -> [('upm', 'Unidad Primaria de Muestreo')]
instru -> []

ciudad: prefijos de 2 dígitos -> ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24']

conglomerado: prefijos de 2 dígitos -> ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '90'

In [7]:
# Verificacion de provincias

print(df["ciudad"].dtype, df["ciudad"].head(5).tolist(), df["ciudad"].nunique())

a_texto = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
df["prov_cod"] = a_texto(df["ciudad"]).str[:2]

tab = df.groupby("prov_cod").agg(n=("fexp", "size"), poblacion=("fexp", "sum"))
tab["poblacion"] = tab["poblacion"].round(0)
print(tab.sort_values("poblacion", ascending=False))

float64 [10150.0, 10150.0, 10150.0, 10150.0, 10150.0] 587
              n  poblacion
prov_cod                  
09        10883  4929602.0
17        11379  3175609.0
14          998  1285310.0
13         1856  1122510.0
08         5269   944843.0
24         3516   930642.0
12         1553   816387.0
07         8591   731693.0
01         7284   729734.0
18         6272   527434.0
05         1564   488165.0
23         1715   457788.0
11         2718   413693.0
22         1736   374768.0
10         3949   337964.0
15         1467   292159.0
06         1427   281800.0
21         2035   239921.0
19         1645   228587.0
03         1472   194336.0
16         1534   192210.0
02         1434   158506.0
04         1858   148849.0
20          739    24270.0


In [8]:
# Construir anios de escolaridad

import pyreadstat
ruta = "../data/raw/spss/enemdu_persona_2026_l_trimestre.sav"
_, meta = pyreadstat.read_sav(ruta, metadataonly=True)

for v in ["p10a", "p10b", "dominio", "estrato"]:
    print(f"\n{v}: {meta.column_names_to_labels.get(v)}")
    lab = meta.variable_value_labels.get(v, "sin etiquetas de valores")
    print(dict(list(lab.items())[:30]) if isinstance(lab, dict) else lab)


p10a: Nivel de instrucción
{1.0: ' Ninguno', 2.0: ' Centro de alfabetización', 3.0: ' Jardín de infantes', 4.0: ' Primaria', 5.0: ' Educación Básica', 6.0: ' Secundaria', 7.0: ' Educación  Media', 8.0: ' Superior no universitario', 9.0: ' Superior Universitario', 10.0: ' Post-grado'}

p10b: Año aprobado
sin etiquetas de valores

dominio: Dominios
{1.0: 'Quito', 2.0: 'Guayaquil', 3.0: 'Cuenca', 4.0: 'Machala', 5.0: 'Ambato', 6.0: 'Resto país'}

estrato: Estratos
sin etiquetas de valores


In [9]:
# Validacion de indicadores por dominio

nombres_dom = {1: "Quito", 2: "Guayaquil", 3: "Cuenca", 4: "Machala", 5: "Ambato"}

calc_dom = pd.DataFrame(
    {nombres_dom[k]: indicadores_laborales(df[df["dominio"] == k]) for k in nombres_dom}
).T.round(1)
calc_dom.columns = ["Participación", "Empleo adecuado", "Subempleo", "Desempleo"]

oficial_dom = pd.DataFrame({
    "Participación":   [61.3, 58.3, 60.8, 62.7, 69.0],
    "Empleo adecuado": [55.7, 46.1, 60.7, 49.1, 43.9],
    "Subempleo":       [11.5, 22.6, 7.7, 14.3, 17.0],
    "Desempleo":       [8.9, 2.3, 4.0, 3.8, 3.7],
}, index=list(nombres_dom.values()))

print("Calculado:\n", calc_dom, "\n")
print("Diferencia (calculado - oficial):\n", (calc_dom - oficial_dom).round(1))

Calculado:
            Participación  Empleo adecuado  Subempleo  Desempleo
Quito               61.3             55.7       11.5        8.9
Guayaquil           58.3             46.1       22.6        2.3
Cuenca              60.8             60.7        7.7        4.0
Machala             62.7             49.1       14.3        3.8
Ambato              69.0             43.9       17.0        3.7 

Diferencia (calculado - oficial):
            Participación  Empleo adecuado  Subempleo  Desempleo
Quito                0.0              0.0        0.0        0.0
Guayaquil            0.0              0.0        0.0        0.0
Cuenca               0.0              0.0        0.0        0.0
Machala              0.0              0.0        0.0        0.0
Ambato               0.0              0.0        0.0        0.0


In [10]:
# Tablas de resultados agregados

import os
os.makedirs("../reports", exist_ok=True)
calc_dom.to_csv("../reports/validacion_dominios.csv", encoding="utf-8-sig")
comparacion.to_csv("../reports/validacion_nacional.csv", encoding="utf-8-sig")